*0.2 Math / ML basics*

# Embedding spaces

**The situation.** Search results puzzle the product manager: a query about "delivery" returns articles about "shipping", "courier" and "tracking" — none contain the word. She asks how the system "knows" they are related. The answer is the *space* the vectors live in.

**An embedding space.** Every text is a point in a 1,536-dimensional space. Training arranged the points so that things used in similar ways are near each other: "shipping" near "delivery", "invoice" near "receipt", and both clusters far from "password". Neighbourhoods in that space are the model's notion of meaning. You cannot see 1,536 dimensions — the next three items squash them to two so you can — but you can query the neighbourhoods directly.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import numpy as np
from openai import OpenAI

client = OpenAI(timeout=30)
words = [
    "delivery",
    "shipping",
    "courier",
    "tracking number",
    "invoice",
    "receipt",
    "refund",
    "payment",
    "password",
    "login",
    "two-factor code",
    "username",
]
vectors = []
for item in client.embeddings.create(model="text-embedding-3-small", input=words).data:
    vectors.append(item.embedding)
vectors = np.array(vectors, dtype=np.float32)
similarity = vectors @ vectors.T  # unit-length, so dot product = cosine

for probe in ("delivery", "invoice", "password"):
    row = similarity[words.index(probe)].copy()
    row[words.index(probe)] = -1  # ignore the word itself
    neighbours = []
    for i in np.argsort(-row)[:3]:
        neighbours.append(f"{words[i]} ({row[i]:.2f})")
    print(f"{probe:<10} → " + ", ".join(neighbours))
assert words[np.argsort(-similarity[0].copy())[1]] in ("shipping", "courier", "tracking number")

delivery   → shipping (0.68), courier (0.52), payment (0.47)
invoice    → payment (0.65), receipt (0.57), refund (0.51)
password   → username (0.66), login (0.54), two-factor code (0.44)


**Reading the output.** "delivery" lives next to shipping, courier and tracking; "invoice" next to receipt, refund, payment; "password" next to login, username, two-factor. Nobody wrote those groupings; they fell out of training on text where those words appear in the same situations.

```
                  · password  · login
                  · username  · two-factor code
                                                     · invoice   · receipt
     · delivery  · shipping                          · payment   · refund
     · courier   · tracking number
```

**The rule to remember.** An embedding space is meaning laid out as geometry. Near = related. Every retrieval, clustering and deduplication system is a question about neighbourhoods in that space.

| Use it when | Don't when | Instead use |
|---|---|---|
| reasoning about what a vector search will and will not find | you need the space to encode facts (dates, prices, ids) — it does not | metadata filters next to the vector search |

**Watch out**
- Antonyms are neighbours: "refund approved" and "refund denied" sit close. Vector search is not a fact checker.
- The space is the model's. A new model version is a new space; never mix vectors across versions.
- Short strings ("ok", "yes") land in noisy places; embed enough context to mean something.